# Fase 5 LoRA — Notebook 00: construir dataset (img + ann + manifest)

Reconstruye el dataset de 600 muestras revisadas en Supervisely **sin tocar Supervisely**.

- Entrada: 2 tars (jobs 185735 + 185736), cada uno 300 ann disjuntas → 600 únicas.
- Las imágenes NO venian en los tars: se **recrean** desde los videos fuente.
  El nombre `IMG_9779_f0036` codifica el **índice de frame fuente 36** (decord directo) → match pixel-perfect.
- Salidas en `fase_5_lora/`:
  - `dataset/testing_600/{ann,img}` (600 + 600), `dataset/meta.json`
  - `dataset/manifest.csv` con split **80/20 por-video** (sin fuga)
  - `Opus/` 50 overlays random + `Opus_review_50.zip` para revisión visual

Clases (meta, incluida la `blue_zone` agregada manual): orange_ball, robot_a, robot_b, green_floor, yellow_zone, blue_zone.

In [1]:
import os, sys, re, json, glob, shutil, tarfile, random, csv
from pathlib import Path
from collections import Counter
import numpy as np
import cv2
import decord
import supervisely as sly

BASE = Path('/workspace/FutBotMX-UAQTeam/notebooks/fase_5_lora')
INCOMING = BASE / '_incoming'
DATASET  = BASE / 'dataset'
DS_NAME  = 'testing_600'
ANN_DIR  = DATASET / DS_NAME / 'ann'
IMG_DIR  = DATASET / DS_NAME / 'img'
META_GLASSES = Path('/workspace/Meta_Glasses')
OPUS = BASE / 'Opus'
SEED = 42
print('decord', decord.__version__, '| sly', sly.__version__)
print('tars:', [p.name for p in sorted(INCOMING.glob('*.tar.gz'))])

decord 0.6.0 | sly 6.73.580
tars: ['job_185735.tar.gz', 'job_185736.tar.gz']


## 1. Extraer los 2 jobs y fusionar anotaciones (600)

In [2]:
tars = sorted(INCOMING.glob('*.tar.gz'))
assert len(tars) >= 2, f'esperaba 2 tars, hay {tars}'

work = BASE / '_extract'
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

roots = []
for t in tars:
    dst = work / t.stem
    dst.mkdir()
    with tarfile.open(t) as tf:
        tf.extractall(dst)
    metas = list(dst.rglob('meta.json'))
    assert metas, f'sin meta.json en {t.name}'
    roots.append(metas[0].parent)
    print('extraido:', t.name, '->', metas[0].parent.name)

# Reset dataset destino
if DATASET.exists():
    shutil.rmtree(DATASET)
ANN_DIR.mkdir(parents=True)
IMG_DIR.mkdir(parents=True)
shutil.copy(roots[0] / 'meta.json', DATASET / 'meta.json')

n = 0
for r in roots:
    ann_src = r / DS_NAME / 'ann'
    if not ann_src.exists():
        ann_src = next(p for p in r.rglob('ann') if p.is_dir())
    for j in ann_src.glob('*.json'):
        shutil.copy(j, ANN_DIR / j.name)
        n += 1

uniq = len(list(ANN_DIR.glob('*.json')))
print(f'ann copiadas: {n} | unicas en destino: {uniq}')
assert n == 600 and uniq == 600, 'esperaba 600 ann unicas'

extraido: job_185735.tar.gz -> FutBot_Testing_600_smoothv3_tta (reviewed by job 185735)


extraido: job_185736.tar.gz -> FutBot_Testing_600_smoothv3_tta (reviewed by job 185736)


ann copiadas: 600 | unicas en destino: 600


## 2. Indexar videos fuente + parseo nombre → (video, frame_idx)

In [3]:
# Mapa stem_video(lower) -> ruta real (recursivo, case-insensitive)
vids = {}
for p in META_GLASSES.rglob('*'):
    if p.is_file() and p.suffix.lower() == '.mov':
        vids[p.stem.lower()] = p
print('videos .mov indexados en Meta_Glasses:', len(vids))

PAT = re.compile(r'^(?P<v>.+)_f(?P<i>\d+)$')
def parse_name(json_name):
    """'IMG_9779_f0036.png.json' -> ('IMG_9779', 36)"""
    base = json_name[:-len('.png.json')] if json_name.endswith('.png.json') else Path(json_name).stem
    m = PAT.match(base)
    assert m, f'nombre no parseable: {json_name}'
    return m.group('v'), int(m.group('i'))

needed = sorted({parse_name(j.name)[0] for j in ANN_DIR.glob('*.json')})
missing = [v for v in needed if v.lower() not in vids]
print(f'videos requeridos: {len(needed)} | faltantes: {missing}')
assert not missing, f'faltan videos fuente: {missing}'

videos .mov indexados en Meta_Glasses: 123
videos requeridos: 20 | faltantes: []


## 3. Recrear las 600 imágenes (frame exacto por decord) y verificar match 1:1

In [4]:
readers = {}
def get_reader(stem):
    k = stem.lower()
    if k not in readers:
        readers[k] = decord.VideoReader(str(vids[k]))
    return readers[k]

mismatch = []
made = 0
anns = sorted(ANN_DIR.glob('*.json'))
for j in anns:
    img_name = j.name[:-5]          # quita '.json' -> 'IMG_..._f0000.png'
    v, idx = parse_name(j.name)
    r = get_reader(v)
    assert 0 <= idx < len(r), f'idx {idx} fuera de rango ({len(r)}) en {v}'
    frame = r[idx].asnumpy()        # RGB HxWx3
    d = json.load(open(j, encoding='utf-8'))
    H, W = d['size']['height'], d['size']['width']
    if frame.shape[0] != H or frame.shape[1] != W:
        mismatch.append((img_name, frame.shape[:2], (H, W)))
        frame = cv2.resize(frame, (W, H), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(IMG_DIR / img_name), cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
    made += 1
    if made % 100 == 0:
        print(f'  {made}/600 ...')

print('imagenes recreadas:', made)
print('mismatch de resolucion (re-escaladas):', len(mismatch), mismatch[:5])

img_set = {p.name for p in IMG_DIR.glob('*.png')}
ann_set = {p.name[:-5] for p in ANN_DIR.glob('*.json')}
print('img==ann por nombre:', img_set == ann_set, '| img', len(img_set), '| ann', len(ann_set))
assert img_set == ann_set, 'desajuste img/ann'

  100/600 ...


  200/600 ...


  300/600 ...


  400/600 ...


  500/600 ...


  600/600 ...
imagenes recreadas: 600
mismatch de resolucion (re-escaladas): 0 []
img==ann por nombre: True | img 600 | ann 600


## 4. Validar anotaciones (conteo por clase, blue_zone, vacías)

In [5]:
meta = json.load(open(DATASET / 'meta.json', encoding='utf-8'))
meta_classes = [c['title'] for c in meta['classes']]

obj_count = Counter()
img_with = Counter()
empty = []
for j in sorted(ANN_DIR.glob('*.json')):
    d = json.load(open(j, encoding='utf-8'))
    objs = d.get('objects', [])
    if not objs:
        empty.append(j.name)
    seen = set()
    for o in objs:
        obj_count[o['classTitle']] += 1
        seen.add(o['classTitle'])
    for c in seen:
        img_with[c] += 1

print('clases en meta:', meta_classes)
print(f"{'clase':14s} {'#objetos':>9s} {'#imgs':>7s}")
for c in meta_classes:
    print(f'{c:14s} {obj_count.get(c,0):9d} {img_with.get(c,0):7d}')
extra = set(obj_count) - set(meta_classes)
print('clases fuera de meta:', extra)
print('blue_zone presente:', 'blue_zone' in obj_count, '->', obj_count.get('blue_zone', 0), 'objetos en', img_with.get('blue_zone',0), 'imgs')
print('anns vacias:', len(empty), empty[:5])
assert not extra, f'clases inesperadas: {extra}'

clases en meta: ['orange_ball', 'robot_a', 'robot_b', 'green_floor', 'yellow_zone', 'blue_zone']
clase           #objetos   #imgs
orange_ball          419     418
robot_a              818     538
robot_b              623     467
green_floor          630     600
yellow_zone          550     515
blue_zone            404     401
clases fuera de meta: set()
blue_zone presente: True -> 404 objetos en 401 imgs
anns vacias: 0 []


## 5. Manifest CSV — split 80/20 por-video (sin fuga)

In [6]:
rng = random.Random(SEED)
videos = sorted({parse_name(j.name)[0] for j in ANN_DIR.glob('*.json')})
shuffled = videos[:]
rng.shuffle(shuffled)
n_test = max(1, round(len(shuffled) * 0.2))
test_videos = set(shuffled[:n_test])

rows = []
for j in sorted(ANN_DIR.glob('*.json')):
    img_name = j.name[:-5]
    v, idx = parse_name(j.name)
    rows.append({
        'image': f'{DS_NAME}/img/{img_name}',
        'ann':   f'{DS_NAME}/ann/{j.name}',
        'video': v,
        'frame_idx': idx,
        'split': 'test' if v in test_videos else 'train',
    })

csv_path = DATASET / 'manifest.csv'
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=['image', 'ann', 'video', 'frame_idx', 'split'])
    w.writeheader()
    w.writerows(rows)

tr = sum(r['split'] == 'train' for r in rows)
te = len(rows) - tr
print('manifest:', csv_path)
print(f'train: {tr} frames ({len(videos)-n_test} videos) | test: {te} frames ({n_test} videos)')
print('videos TEST:', sorted(test_videos))

manifest: /workspace/FutBotMX-UAQTeam/notebooks/fase_5_lora/dataset/manifest.csv
train: 480 frames (16 videos) | test: 120 frames (4 videos)
videos TEST: ['IMG_9854', 'IMG_9855', 'video-488_singular_display', 'video-836_singular_display']


## 6. Carpeta Opus — 50 overlays random (raw | overlay translucido + leyenda) + ZIP

In [7]:
def hex2bgr(h):
    h = h.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return (b, g, r)

color = {c['title']: hex2bgr(c['color']) for c in meta['classes']}

def render_overlay(img_name, alpha=0.45):
    img = cv2.imread(str(IMG_DIR / img_name))            # BGR
    d = json.load(open(ANN_DIR / (img_name + '.json'), encoding='utf-8'))
    ov = img.copy()
    for o in d['objects']:
        m = sly.Bitmap.base64_2_data(o['bitmap']['data'])  # bool sub-mask
        ox, oy = o['bitmap']['origin']                     # col, row
        h = min(m.shape[0], ov.shape[0] - oy)
        w = min(m.shape[1], ov.shape[1] - ox)
        roi = ov[oy:oy+h, ox:ox+w]
        mm = m[:h, :w]
        roi[mm] = np.array(color.get(o['classTitle'], (255, 255, 255)), dtype=np.uint8)
    return cv2.addWeighted(ov, alpha, img, 1 - alpha, 0)

def add_legend(img):
    leg = np.full((img.shape[0], 230, 3), 35, np.uint8)
    y = 30
    for name, c in color.items():
        cv2.rectangle(leg, (12, y-16), (44, y+4), tuple(int(x) for x in c), -1)
        cv2.putText(leg, name, (52, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
        y += 34
    return np.hstack([img, leg])

rng2 = random.Random(SEED)
all_imgs = sorted(IMG_DIR.glob('*.png'))
sample = rng2.sample(all_imgs, min(50, len(all_imgs)))

if OPUS.exists():
    shutil.rmtree(OPUS)
OPUS.mkdir(parents=True)
for p in sample:
    raw = cv2.imread(str(p))
    ov = render_overlay(p.name)
    combo = add_legend(np.hstack([raw, ov]))
    cv2.imwrite(str(OPUS / f'{p.stem}__raw_vs_overlay.png'), combo)

zip_base = str(BASE / 'Opus_review_50')
if os.path.exists(zip_base + '.zip'):
    os.remove(zip_base + '.zip')
shutil.make_archive(zip_base, 'zip', OPUS)
sz = os.path.getsize(zip_base + '.zip') / 1e6
print('Opus:', OPUS, '| muestras:', len(sample))
print('ZIP:', zip_base + '.zip', f'-> {sz:.1f} MB')

Opus: /workspace/FutBotMX-UAQTeam/notebooks/fase_5_lora/Opus | muestras: 50
ZIP: /workspace/FutBotMX-UAQTeam/notebooks/fase_5_lora/Opus_review_50.zip -> 187.7 MB


## 7. Resumen

Artefactos generados en `notebooks/fase_5_lora/`:
- `dataset/meta.json` — 6 clases (incl. `blue_zone`).
- `dataset/testing_600/ann/*.json` — 600 anotaciones (merge de los 2 jobs).
- `dataset/testing_600/img/*.png` — 600 imágenes recreadas (match 1:1).
- `dataset/manifest.csv` — split train/test 80/20 por-video.
- `Opus/` + `Opus_review_50.zip` — 50 overlays para revisión visual.

Supervisely no se modificó en ningún momento.